# 5. Main Regression Analysis (R): Wealth x Cardiometabolic Burden Interaction

This is the R replication of `5. Main Regression Analysis.ipynb`, using the `survey` package's
`svydesign()` / `svyglm()` for genuine design-based (stratified, clustered, weighted) inference --
the piece the Python version could only approximate.

**Why this notebook exists alongside the Python one, rather than replacing it:** the Python
notebook was the right tool for model-building, iterating on the sparse-cell diagnosis, and
prototyping the covariate set quickly. This notebook takes those exact same model specifications
(same formulas, same reference categories, same 12 covariates) and re-fits them with a proper
`svydesign(id = ~PSU, strata = ~Stratum, weights = ~Sampling.weight, nest = TRUE)`, which:

- correctly incorporates **stratification** (`statsmodels` has no equivalent -- clustering-only)
- uses `anova(m1, m2, method = "LRT")`, a genuine **design-based Rao-Scott working LRT**, not an
  unadjusted deviance-difference test
- replaces the Python stratified-cluster-bootstrap cross-check (a workaround) with the real thing

**Point estimates (odds ratios) should match the Python notebook closely** -- both maximize the
same weighted pseudo-likelihood. **Standard errors, confidence intervals, and the nested-model
p-values are what actually change**, and these are the numbers that should go in the manuscript.

In [1]:
suppressMessages(library(survey))

dep <- read.csv("../resources/depression_dataset.csv")
anx <- read.csv("../resources/anxiety_dataset.csv")

cat("Depression analytic sample:", nrow(dep), "rows,", ncol(dep), "cols\n")
cat("Anxiety analytic sample:   ", nrow(anx), "rows,", ncol(anx), "cols\n")

stopifnot(nrow(dep) == 4887, nrow(anx) == 4887)


Depression analytic sample: 4887 rows, 27 cols


Anxiety analytic sample:    4887 rows, 27 cols


## 2. Outcome definition

Identical to the Python notebook: probable depression = PHQ-9 >= 10 (`Depression` band >= 2);
probable anxiety = GAD-7 >= 10 (`Anxiety` band >= 2). Kroenke, Spitzer & Williams (2001) and
Spitzer, Kroenke, Williams & Lowe (2006) respectively -- see the Python notebook Section 2 for
the full citation and the caveat that this cutoff is a documented assumption, not a given.

In [2]:
dep$Depression_binary <- as.integer(dep$Depression >= 2)
anx$Anxiety_binary <- as.integer(anx$Anxiety >= 2)

cat(sprintf("Depression: %d probable cases / %d (%.1f%%)\n",
            sum(dep$Depression_binary), nrow(dep), 100*mean(dep$Depression_binary)))
cat(sprintf("Anxiety:    %d probable cases / %d (%.1f%%)\n",
            sum(anx$Anxiety_binary), nrow(anx), 100*mean(anx$Anxiety_binary)))


Depression: 235 probable cases / 4887 (4.8%)


Anxiety:    221 probable cases / 4887 (4.5%)


## 3. Variable coding and survey design object

Reference categories match the Python notebook exactly (Richest for Wealth, otherwise the
substantively natural baseline). `read.csv()` mangles the space-containing column names from the
CSV (e.g. `Socioeconomic Status` -> `Socioeconomic.Status`) -- handled below.

The `svydesign()` call is the whole point of this notebook: `nest = TRUE` tells `survey` that PSU
codes are only unique *within* stratum (true here -- BDHS PSU numbering restarts per stratum),
which is exactly the design detail `statsmodels` had no way to express.

In [3]:
prep <- function(df) {
  df$SES     <- relevel(factor(df$Socioeconomic.Status), ref = "5")   # ref = Richest
  df$Burden_num <- df$Cardiometabolic.Burden
  df$Burden_collapsed <- factor(ifelse(df$Cardiometabolic.Burden >= 2, 2, df$Cardiometabolic.Burden))

  df$Education          <- relevel(factor(df$Education), ref = "0")            # No education
  df$Occupation         <- relevel(factor(df$Occupation), ref = "0")           # No
  df$Partner.occupation <- relevel(factor(df$Partner.occupation), ref = "2")   # Working
  df$Age                <- relevel(factor(df$Age), ref = "1")                  # 15-24
  df$Division            <- relevel(factor(df$Division), ref = "3")            # Dhaka
  df$Residence           <- relevel(factor(df$Residence), ref = "1")           # Urban
  df$Religion             <- relevel(factor(df$Religion), ref = "1")           # Islam
  df$Children             <- relevel(factor(df$Children), ref = "0")           # No children
  df$Family.size          <- relevel(factor(df$Family.size), ref = "1")        # <5 members
  df$Household.Autonomy   <- relevel(factor(df$Household.Autonomy), ref = "0") # No autonomy
  df$Insurance             <- relevel(factor(df$Insurance), ref = "0")         # No
  df$Internet              <- relevel(factor(df$Internet), ref = "0")         # Never
  df
}

dep <- prep(dep)
anx <- prep(anx)

des_dep <- svydesign(id = ~PSU, strata = ~Stratum, weights = ~Sampling.weight, data = dep, nest = TRUE)
des_anx <- svydesign(id = ~PSU, strata = ~Stratum, weights = ~Sampling.weight, data = anx, nest = TRUE)

cat("Survey design objects created.\n")
summary(des_dep)


Survey design objects created.


Stratified 1 - level Cluster Sampling design (with replacement)
With (674) clusters.
svydesign(id = ~PSU, strata = ~Stratum, weights = ~Sampling.weight, 
    data = dep, nest = TRUE)
Probabilities:
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  0.257   0.782   1.059   1.447   1.522  10.950 
Stratum Sizes: 
             1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16
obs        178 370 298 438 342 358 199 429 126 407 218 403 177 409 171 364
design.PSU  24  47  39  55  52  53  28  58  17  58  31  58  23  61  23  47
actual.PSU  24  47  39  55  52  53  28  58  17  58  31  58  23  61  23  47
Data variables:
 [1] "Cardiometabolic.Burden"    "Socioeconomic.Status"     
 [3] "Depression"                "Education"                
 [5] "Occupation"                "Partner.occupation"       
 [7] "Age"                       "Division"                 
 [9] "Residence"                 "Religion"                 
[11] "Children"                  "Family.size"              
[13] "

## 4. Wealth x Burden sparse-cell check

Same finding as the Python notebook: `Poorest x 3 burdens` has n = 1 (uncollapsed). Model 2 uses
linear Burden (parsimonious, pools information); Model 3 uses the collapsed 0/1/2+ Burden coding
as a sensitivity check that relaxes linearity without hitting the n=1 cell.

In [4]:
cat("Wealth x Cardiometabolic Burden (uncollapsed):\n")
print(table(dep$SES, dep$Cardiometabolic.Burden))
cat("\nWealth x Burden_collapsed (0 / 1 / 2+):\n")
ct <- table(dep$SES, dep$Burden_collapsed)
print(ct)
cat("\nMinimum cell count:", min(ct), "\n")


Wealth x Cardiometabolic Burden (uncollapsed):


   
      0   1   2   3
  5 710 290 108  16
  1 687 145  26   1
  2 733 175  30   6
  3 725 173  40   6
  4 716 217  76   7



Wealth x Burden_collapsed (0 / 1 / 2+):


   
      0   1   2
  5 710 290 124
  1 687 145  27
  2 733 175  36
  3 725 173  46
  4 716 217  83



Minimum cell count: 27 


## 5. Model formulas and a tidy odds-ratio helper

In [5]:
covariates <- c("Education", "Occupation", "Partner.occupation", "Age", "Division",
                "Residence", "Religion", "Children", "Family.size", "Household.Autonomy",
                "Insurance", "Internet")
cov_formula <- paste(covariates, collapse = " + ")

f_m1  <- function(outcome) as.formula(paste(outcome, "~ SES + Burden_num +", cov_formula))
f_m2  <- function(outcome) as.formula(paste(outcome, "~ SES * Burden_num +", cov_formula))
# Model 3's *main-effect* comparator must use the same Burden_collapsed parameterization as
# Model 3 itself -- Model 1 (Burden_num) is NOT nested inside Model 3 (Burden_collapsed) because
# they encode the moderator differently, not just with/without an interaction. R's anova.svyglm
# correctly refuses this comparison ("models not nested"); statsmodels' plain deviance-difference
# test in the Python notebook did not catch this and should not be trusted for that comparison.
f_m1c <- function(outcome) as.formula(paste(outcome, "~ SES + Burden_collapsed +", cov_formula))
f_m3  <- function(outcome) as.formula(paste(outcome, "~ SES * Burden_collapsed +", cov_formula))

or_table <- function(model, label = "") {
  co <- coef(model); ci <- confint(model); se <- summary(model)$coefficients[, "Std. Error"]
  p  <- summary(model)$coefficients[, "Pr(>|t|)"]
  out <- data.frame(
    term = names(co), OR = exp(co), CI_low = exp(ci[, 1]), CI_high = exp(ci[, 2]), p = p
  )
  out <- out[out$term != "(Intercept)", ]
  rownames(out) <- NULL
  if (nchar(label) > 0) out <- cbind(model = label, out)
  out
}

simple_slopes <- function(model, wealth_prefix, burden_var, burden_values, wealth_levels) {
  V <- vcov(model); b <- coef(model)
  rows <- list()
  for (level in wealth_levels) {
    main_term <- paste0(wealth_prefix, level)
    int_term  <- paste0(wealth_prefix, level, ":", burden_var)
    if (!(main_term %in% names(b))) next
    for (bv in burden_values) {
      beta <- b[main_term] + ifelse(int_term %in% names(b), bv * b[int_term], 0)
      var_ <- V[main_term, main_term]
      if (int_term %in% rownames(V)) {
        var_ <- var_ + bv^2 * V[int_term, int_term] + 2 * bv * V[main_term, int_term]
      }
      se <- sqrt(var_)
      or_ <- exp(beta); lo <- exp(beta - 1.96 * se); hi <- exp(beta + 1.96 * se)
      z <- beta / se; pval <- 2 * (1 - pnorm(abs(z)))
      rows[[length(rows) + 1]] <- data.frame(Wealth_vs_Richest = level, Burden = bv,
                                              OR = or_, CI_low = lo, CI_high = hi, p = pval)
    }
  }
  do.call(rbind, rows)
}

round_df <- function(df, digits = 3) {
  num_cols <- sapply(df, is.numeric)
  df[num_cols] <- lapply(df[num_cols], round, digits = digits)
  df
}
cat("Helpers defined.\n")


Helpers defined.


## 6. Depression models (design-based)

In [6]:
m1_dep  <- svyglm(f_m1("Depression_binary"),  design = des_dep, family = quasibinomial())
m2_dep  <- svyglm(f_m2("Depression_binary"),  design = des_dep, family = quasibinomial())
m1c_dep <- svyglm(f_m1c("Depression_binary"), design = des_dep, family = quasibinomial())
m3_dep  <- svyglm(f_m3("Depression_binary"),  design = des_dep, family = quasibinomial())

cat("Depression Model 2 (primary): Wealth x Burden odds ratios\n")
print(round_df(or_table(m2_dep)), row.names = FALSE)


Depression Model 2 (primary): Wealth x Burden odds ratios


                term    OR CI_low CI_high     p
                SES1 1.597  0.819   3.114 0.169
                SES2 2.042  1.085   3.841 0.027
                SES3 1.787  0.916   3.486 0.088
                SES4 1.647  0.885   3.067 0.115
          Burden_num 1.383  0.895   2.138 0.144
          Education1 0.910  0.597   1.386 0.660
          Education2 0.850  0.543   1.332 0.479
          Education3 0.473  0.255   0.878 0.018
         Occupation1 0.923  0.664   1.282 0.631
 Partner.occupation1 1.182  0.484   2.882 0.713
 Partner.occupation3 0.000  0.000   0.000 0.000
                Age2 1.593  0.963   2.635 0.070
                Age3 2.020  1.178   3.465 0.011
           Division1 1.359  0.755   2.447 0.306
           Division2 1.119  0.602   2.080 0.721
           Division4 1.596  0.952   2.677 0.076
           Division5 1.039  0.564   1.915 0.901
           Division6 0.839  0.450   1.567 0.582
           Division7 1.625  0.881   2.995 0.120
           Division8 1.040  0.531   2.03

**Design-based nested model tests** -- this is the number that should replace the Python
notebook's unadjusted deviance-difference LR test.

In [7]:
cat("=== Model 1 vs Model 2 (linear Burden interaction), Depression ===\n")
print(anova(m1_dep, m2_dep, method = "LRT"))

cat("\n=== Model 1c vs Model 3 (collapsed-Burden interaction), Depression ===\n")
cat("(Model 1c uses the same Burden_collapsed main effect as Model 3 -- the properly nested comparator)\n")
print(anova(m1c_dep, m3_dep, method = "LRT"))


=== Model 1 vs Model 2 (linear Burden interaction), Depression ===


Working (Rao-Scott+F) LRT for SES:Burden_num
 in svyglm(formula = f_m2("Depression_binary"), design = des_dep, 
    family = quasibinomial())
Working 2logLR =  4.867699 p= 0.29224 
(scale factors:  1.5 0.96 0.8 0.72 );  denominator df= 621



=== Model 1c vs Model 3 (collapsed-Burden interaction), Depression ===


(Model 1c uses the same Burden_collapsed main effect as Model 3 -- the properly nested comparator)


Working (Rao-Scott+F) LRT for SES:Burden_collapsed
 in svyglm(formula = f_m3("Depression_binary"), design = des_dep, 
    family = quasibinomial())
Working 2logLR =  17.90066 p= 0.028254 
(scale factors:  1.5 1.3 1.1 1.1 0.95 0.94 0.78 0.35 );  denominator df= 616


### Depression Model 3: interaction terms

**Correction to the Python notebook:** its "Model 1 vs Model 3" test (p = 0.012) compared models
that were not actually nested -- Model 1 used `Burden_num` (linear) while Model 3 used
`Burden_collapsed` (categorical), so a plain deviance-difference test between them isn't valid.
`statsmodels` didn't catch this; R's `anova.svyglm` refused to run it at all ("models not
nested"), which is what surfaced the issue. The corrected comparison above (Model 1c vs Model 3,
both using `Burden_collapsed`) is the valid version of that test -- compare its p-value to
Python's 0.012 rather than treating them as the same test with different SEs.

The interaction terms below show which wealth/burden combination is driving whatever the
corrected test finds -- including the `Wealth=Richer x Burden(2+)` term Python flagged at
OR = 0.028, p = 0.001, as needing a stability check.

In [8]:
interaction_terms_dep <- or_table(m3_dep)
interaction_terms_dep <- interaction_terms_dep[grepl(":", interaction_terms_dep$term), ]
print(round_df(interaction_terms_dep), row.names = FALSE)


                   term    OR CI_low CI_high     p
 SES1:Burden_collapsed1 1.035  0.277   3.872 0.959
 SES2:Burden_collapsed1 1.137  0.383   3.381 0.817
 SES3:Burden_collapsed1 0.683  0.164   2.849 0.600
 SES4:Burden_collapsed1 2.023  0.563   7.264 0.280
 SES1:Burden_collapsed2 0.596  0.102   3.490 0.565
 SES2:Burden_collapsed2 0.146  0.016   1.308 0.085
 SES3:Burden_collapsed2 0.491  0.100   2.411 0.380
 SES4:Burden_collapsed2 0.028  0.003   0.250 0.001


## 7. Simple slopes: Wealth OR at each Burden level (Depression)

Design-based delta method using `vcov(m2_dep)`, which already reflects the full stratified
cluster design -- no separate bootstrap needed here (unlike the Python notebook).

In [9]:
dep_slopes <- simple_slopes(m2_dep, "SES", "Burden_num", 0:3, c(1, 2, 3, 4))
print(round_df(dep_slopes), row.names = FALSE)


 Wealth_vs_Richest Burden    OR CI_low CI_high     p
                 1      0 1.597  0.820   3.110 0.168
                 1      1 1.027  0.465   2.270 0.947
                 1      2 0.660  0.151   2.892 0.582
                 1      3 0.425  0.044   4.061 0.457
                 2      0 2.042  1.087   3.837 0.027
                 2      1 0.993  0.500   1.971 0.984
                 2      2 0.483  0.140   1.663 0.249
                 2      3 0.235  0.035   1.555 0.133
                 3      0 1.787  0.917   3.482 0.088
                 3      1 1.086  0.504   2.338 0.833
                 3      2 0.660  0.147   2.952 0.587
                 3      3 0.401  0.039   4.108 0.441
                 4      0 1.647  0.886   3.064 0.115
                 4      1 0.879  0.447   1.727 0.708
                 4      2 0.469  0.144   1.530 0.209
                 4      3 0.250  0.042   1.502 0.130


## 8. Anxiety models (design-based)

In [10]:
m1_anx  <- svyglm(f_m1("Anxiety_binary"),  design = des_anx, family = quasibinomial())
m2_anx  <- svyglm(f_m2("Anxiety_binary"),  design = des_anx, family = quasibinomial())
m1c_anx <- svyglm(f_m1c("Anxiety_binary"), design = des_anx, family = quasibinomial())
m3_anx  <- svyglm(f_m3("Anxiety_binary"),  design = des_anx, family = quasibinomial())

cat("Anxiety Model 2 (primary): Wealth x Burden odds ratios\n")
print(round_df(or_table(m2_anx)), row.names = FALSE)

cat("\n=== Model 1 vs Model 2 (linear Burden interaction), Anxiety ===\n")
print(anova(m1_anx, m2_anx, method = "LRT"))

cat("\n=== Model 1c vs Model 3 (collapsed-Burden interaction), Anxiety ===\n")
cat("(Model 1c uses the same Burden_collapsed main effect as Model 3 -- the properly nested comparator)\n")
print(anova(m1c_anx, m3_anx, method = "LRT"))


Anxiety Model 2 (primary): Wealth x Burden odds ratios


                term    OR CI_low CI_high     p
                SES1 1.001  0.464   2.161 0.998
                SES2 1.302  0.680   2.493 0.426
                SES3 1.134  0.560   2.296 0.727
                SES4 1.337  0.715   2.503 0.363
          Burden_num 0.999  0.608   1.643 0.997
          Education1 0.855  0.543   1.344 0.496
          Education2 0.939  0.612   1.441 0.772
          Education3 0.472  0.221   1.008 0.053
         Occupation1 0.922  0.654   1.300 0.643
 Partner.occupation1 1.336  0.648   2.754 0.432
 Partner.occupation3 1.457  0.162  13.076 0.736
                Age2 1.259  0.733   2.163 0.404
                Age3 1.912  1.084   3.371 0.025
           Division1 1.156  0.636   2.103 0.634
           Division2 1.337  0.762   2.345 0.311
           Division4 1.401  0.822   2.386 0.214
           Division5 0.774  0.411   1.457 0.427
           Division6 0.701  0.361   1.359 0.292
           Division7 1.443  0.791   2.632 0.231
           Division8 0.946  0.510   1.75


=== Model 1 vs Model 2 (linear Burden interaction), Anxiety ===


Working (Rao-Scott+F) LRT for SES:Burden_num
 in svyglm(formula = f_m2("Anxiety_binary"), design = des_anx, family = quasibinomial())
Working 2logLR =  5.105252 p= 0.26899 
(scale factors:  1.5 1 0.75 0.71 );  denominator df= 621



=== Model 1c vs Model 3 (collapsed-Burden interaction), Anxiety ===


(Model 1c uses the same Burden_collapsed main effect as Model 3 -- the properly nested comparator)


Working (Rao-Scott+F) LRT for SES:Burden_collapsed
 in svyglm(formula = f_m3("Anxiety_binary"), design = des_anx, family = quasibinomial())
Working 2logLR =  8.957388 p= 0.3397 
(scale factors:  2 1.4 1.2 1 0.97 0.87 0.57 9.4e-07 );  denominator df= 616


In [11]:
anx_slopes <- simple_slopes(m2_anx, "SES", "Burden_num", 0:3, c(1, 2, 3, 4))
print(round_df(anx_slopes), row.names = FALSE)


 Wealth_vs_Richest Burden    OR CI_low CI_high     p
                 1      0 1.001  0.464   2.158 0.998
                 1      1 0.448  0.164   1.222 0.117
                 1      2 0.201  0.030   1.342 0.098
                 1      3 0.090  0.005   1.631 0.103
                 2      0 1.302  0.681   2.490 0.425
                 2      1 0.593  0.261   1.348 0.212
                 2      2 0.270  0.056   1.304 0.103
                 2      3 0.123  0.011   1.378 0.089
                 3      0 1.134  0.561   2.292 0.727
                 3      1 1.131  0.548   2.333 0.739
                 3      2 1.128  0.288   4.420 0.863
                 3      3 1.125  0.134   9.427 0.913
                 4      0 1.337  0.715   2.500 0.362
                 4      1 1.130  0.612   2.086 0.695
                 4      2 0.955  0.321   2.846 0.934
                 4      3 0.807  0.150   4.356 0.803


## 9. Combined summary: design-based interaction tests, both outcomes

In [12]:
lr_dep <- anova(m1_dep, m2_dep, method = "LRT")
lr_anx <- anova(m1_anx, m2_anx, method = "LRT")

summary_df <- data.frame(
  Outcome = c("Depression", "Anxiety"),
  N = c(nrow(dep), nrow(anx)),
  Working_chisq = round(c(lr_dep$chisq, lr_anx$chisq), 2),
  df = c(lr_dep$df, lr_anx$df),
  p_value = round(c(lr_dep$p, lr_anx$p), 4)
)
print(summary_df, row.names = FALSE)


    Outcome    N Working_chisq df p_value
 Depression 4887          6.29  4  0.2922
    Anxiety 4887          5.78  4  0.2690


## 10. Interpretation and what changed vs. the Python notebook

Compare the `p_value` column above to the Python notebook's LR test output
(Depression linear: p = 0.182; Anxiety linear: p = 0.220). If the design-based p-values here are
close to those, the Python cluster-robust approximation wasn't misleading for the main interaction
test -- if they diverge meaningfully, **these are the numbers to report**, not Python's.

Everything else about interpreting the output -- read the omnibus test before individual simple
slopes, treat the Model 3 sensitivity check as evidence about the linearity assumption rather
than a separate finding, and confirm the PHQ-9/GAD-7 cutoff and `Financial Decision-Making`
question before finalizing -- carries over unchanged from the Python notebook's Section 12.

**Remaining open item:** the Python notebook flagged `Wealth=Richer x Burden(2+)` (OR = 0.028)
as needing a separation/stability check. Compare that term's row in Section 6 above -- if the
design-based CI is similarly extreme *and* stable (not blown up further), that strengthens
confidence it's real; if the design-based SE is wildly different, treat the Python estimate as
an artifact of the unvalidated cluster+weights covariance and prioritize this R output instead.